# Part 2: Streaming application using Spark Structured Streaming  
In this task, you will implement Spark Structured Streaming to consume the data from task 1 and perform a prediction.    
Important:   
-	This task uses PySpark Structured Streaming with PySpark Dataframe APIs and PySpark ML.  
-	You also need your pipeline model from A2A to make predictions and persist the results.  
  
***Name : Muhammad Iqbal (35092882)***

In [1]:
# Import Library
# Spark Configuration
from pyspark import SparkConf
from pyspark.sql import SparkSession

# Data Schema
from pyspark.sql.types import *
from pyspark.ml.linalg import VectorUDT

# PySpark Data Manipulation
from pyspark.sql.functions import (
        col, from_json, explode, from_unixtime, concat_ws, to_timestamp, window,
        hour, to_date, when, avg, first, broadcast, current_timestamp, to_json,
        struct
    )
from pyspark.sql import functions as F, types as T
import time, shutil, os, glob
from typing import Tuple

# Pipeline
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.linalg import Vectors
from pyspark.ml.regression import GBTRegressor

## 1.	Write code to create a SparkSession, which:  
    1) uses four cores with a proper application name;  
    2) use the Melbourne timezone;  
    3) ensure a checkpoint location has been set.

In [2]:
# local[*]: run Spark in local mode with working processors as logical cores on your machine
master = "local[4]"

# The `appName` field is a name to be shown on the Spark cluster UI page
app_name = "Assignment-2B"

# SparkSession Setup
spark = (
    SparkSession.builder
        .master(master)
        .appName(app_name)
        
        # Add Kafka package
        .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
        
        # Set timezone for event-time handling
        .config("spark.sql.session.timeZone", "Australia/Melbourne")
        
        # Set checkpoint directory
        .config("spark.sql.streaming.checkpointLocation", "file:///tmp/spark-checkpoint")
        .getOrCreate()
)

# Spark Logging
sc = spark.sparkContext
sc.setLogLevel('ERROR')

# Kafka Producer
hostip = "kafka"
topic = "weather5s"

## 2.	Write code to define the data schema for the data files, following the data types suggested in the metadata file. Load the static datasets (e.g. building information) into data frames. (You can reuse your code from 2A.)

### ***Data Schema***

In [12]:
# 1. Meters
meters_schema = StructType([
    StructField("building_id",        IntegerType(),  False),
    StructField("meter_type",         StringType(),   True),
    StructField("ts",                 TimestampType(),True),
    StructField("value",              FloatType(),    True),
    StructField("row_id",             IntegerType(),  False),
])

# 2. Building Information Schema
building_information_schema = StructType([
    StructField("site_id",            IntegerType(),  False),
    StructField("building_id",        IntegerType(),  False),
    StructField("primary_use",        StringType(),   True),
    StructField("square_feet",        IntegerType(),  True),
    StructField("floor_count",        IntegerType(),  True),
    StructField("row_id",             IntegerType(),  False),
    StructField("year_built",         IntegerType(),  True),
    StructField("latent_y",           FloatType(),    True),
    StructField("latent_s",           FloatType(),    True),
    StructField("latent_r",           FloatType(),    True),
])

# 3. Weather
weather_schema = StructType([
    StructField("site_id",            IntegerType(),   False),
    StructField("timestamp",          TimestampType(), True),
    StructField("air_temperature",    FloatType(),     True),
    StructField("cloud_coverage",     IntegerType(),   True), 
    StructField("dew_temperature",    FloatType(),     True),
    StructField("sea_level_pressure", FloatType(),     True),
    StructField("wind_direction",     IntegerType(),   True),
    StructField("wind_speed",         FloatType(),     True),

])

### ***Static Datasets***

In [13]:
# 1. Meters
meters_df = (
    spark.read
         .option("header", "true")
         .schema(meters_schema)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss[.SSS]")
         .csv("new_meters.csv")
)

# 2. Building Information
building_df = (
    spark.read
         .option("header", "true")
         .schema(building_information_schema)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss[.SSS]")
         .csv("new_building_information.csv")
)

# 3. Weather
weather_static_df = (
    spark.read
         .option("header", "true")
         .schema(weather_schema)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss[.SSS]")
         .csv("weather.csv")
)

### ***Datasets Loaded Schema Check***

***Meters***

In [5]:
# Preview Records
meters_df.show(2, truncate=False, vertical=True)

-RECORD 0--------------------------
 building_id | 163                 
 meter_type  | c                   
 ts          | 2022-01-01 00:00:00 
 value       | 4.5719              
 row_id      | 3                   
-RECORD 1--------------------------
 building_id | 170                 
 meter_type  | c                   
 ts          | 2022-01-01 00:00:00 
 value       | 11.2891             
 row_id      | 8                   
only showing top 2 rows



In [5]:
# Data Schema
meters_df.printSchema()

root
 |-- building_id: integer (nullable = true)
 |-- meter_type: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- value: float (nullable = true)
 |-- row_id: integer (nullable = true)



***Building***

In [7]:
# Preview Records
building_df.show(2, truncate=False, vertical=True)

-RECORD 0-----------------
 site_id     | 10         
 building_id | 1017       
 primary_use | Technology 
 square_feet | 109263     
 floor_count | 6          
 row_id      | 1018       
 year_built  | 1971       
 latent_y    | 29.0       
 latent_s    | 4.26031    
 latent_r    | 4.0        
-RECORD 1-----------------
 site_id     | 4          
 building_id | 587        
 primary_use | Technology 
 square_feet | 53234      
 floor_count | 5          
 row_id      | 588        
 year_built  | 1949       
 latent_y    | 51.0       
 latent_s    | 4.0271864  
 latent_r    | 3.0        
only showing top 2 rows



In [6]:
# Data Schema
building_df.printSchema()

root
 |-- site_id: integer (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- primary_use: string (nullable = true)
 |-- square_feet: integer (nullable = true)
 |-- floor_count: integer (nullable = true)
 |-- row_id: integer (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- latent_y: float (nullable = true)
 |-- latent_s: float (nullable = true)
 |-- latent_r: float (nullable = true)



***Weather Static***

In [9]:
# Preview Records
weather_static_df.show(2, truncate=False, vertical=True)

-RECORD 0---------------------------------
 site_id            | 0                   
 timestamp          | 2022-01-01 22:00:00 
 air_temperature    | 26.7                
 cloud_coverage     | NULL                
 dew_temperature    | 18.3                
 sea_level_pressure | 1016.9              
 wind_direction     | NULL                
 wind_speed         | 3.1                 
-RECORD 1---------------------------------
 site_id            | 0                   
 timestamp          | 2022-01-01 23:00:00 
 air_temperature    | 25.6                
 cloud_coverage     | NULL                
 dew_temperature    | 18.3                
 sea_level_pressure | 1017.5              
 wind_direction     | NULL                
 wind_speed         | 3.1                 
only showing top 2 rows



In [7]:
# Data Schema
weather_static_df.printSchema()

root
 |-- site_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- air_temperature: float (nullable = true)
 |-- cloud_coverage: integer (nullable = true)
 |-- dew_temperature: float (nullable = true)
 |-- sea_level_pressure: float (nullable = true)
 |-- wind_direction: integer (nullable = true)
 |-- wind_speed: float (nullable = true)



All the static datasets (`new_meters.csv`, `new_building_info.csv`, `weather.csv`) are already load in the notebook with the predefined schema.

## 3.	Using the Kafka topic from the producer in Task 1, ingest the streaming data into Spark Streaming, assuming all data comes in the String format. Except for the 'weather_ts' column, you shall receive it as an Int type.

### ***Stream Data Schema***

In [14]:
# Weather Dataset Schema for Streaming record
weather_record_schema = StructType([
    StructField("site_id",            StringType(),  True),
    StructField("timestamp",          StringType(),  True),
    StructField("air_temperature",    StringType(),  True),
    StructField("cloud_coverage",     StringType(),  True),
    StructField("dew_temperature",    StringType(),  True),
    StructField("sea_level_pressure", StringType(),  True),
    StructField("wind_direction",     StringType(),  True),
    StructField("wind_speed",         StringType(),  True),
    StructField("ts_send",            StringType(),  True),
    StructField("weather_ts",         IntegerType(), True),
    StructField("weather_ts_iso",     StringType(),  True)
])

### ***Stream Datasets from Task 1***

In [15]:
# Read Stream from Kafka
kafka_stream_df = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", f"{hostip}:9092")
         .option("subscribe", topic)
         .load()
)

# Kafka "value" field is in binary, so cast to string
raw_stream_df = kafka_stream_df.selectExpr("CAST(value AS STRING) as json_string")

# Parse the JSON structure
parsed_df = raw_stream_df.select(from_json(col("json_string"), weather_record_schema).alias("data"))
flat_df   = parsed_df.select("data.*")

In [16]:
# Null-safe Cast
def cast_nullable(colname, dtype):
    return F.when((F.col(colname) == "") | F.col(colname).isNull(), None) \
            .otherwise(F.col(colname)).cast(dtype).alias(colname)

# Convert to Predefined Metadata
weather_stream = (
    flat_df.select(
        cast_nullable("site_id",            IntegerType()),
        to_timestamp("timestamp",           "yyyy-MM-dd HH:mm:ss").alias("timestamp"),
        cast_nullable("air_temperature",    FloatType()),
        cast_nullable("cloud_coverage",     IntegerType()),
        cast_nullable("dew_temperature",    FloatType()),
        cast_nullable("sea_level_pressure", FloatType()),
        cast_nullable("wind_direction",     IntegerType()),
        cast_nullable("wind_speed",         FloatType()),
        cast_nullable("ts_send",            IntegerType()),
        col("weather_ts").cast(IntegerType()).alias("weather_ts"),
        to_timestamp("weather_ts_iso",      "yyyy-MM-dd HH:mm:ss").alias("weather_ts_iso")
    )
    # Temporal features
    .withColumn("hour",  F.hour("timestamp"))
)

***Sanity Check***

In [11]:
# Data Schema
weather_stream.printSchema()

root
 |-- site_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- air_temperature: float (nullable = true)
 |-- cloud_coverage: integer (nullable = true)
 |-- dew_temperature: float (nullable = true)
 |-- sea_level_pressure: float (nullable = true)
 |-- wind_direction: integer (nullable = true)
 |-- wind_speed: float (nullable = true)
 |-- ts_send: integer (nullable = true)
 |-- weather_ts: integer (nullable = true)
 |-- weather_ts_iso: timestamp (nullable = true)
 |-- hour: integer (nullable = true)



In [12]:
# Streaming Connectivity
print("weather_stream.isStreaming =", weather_stream.isStreaming)

weather_stream.isStreaming = True


The dataframe is streaming-enable which can receive data in real-time as long as the Producer send their records.

***Pull Query***

In [16]:
# Stop Similar Query for Re-Running
for q in spark.streams.active:
    if q.name == "weather_stream":
        q.stop()
        
# Clear the old checkpoint folder
!rm -rf /tmp/ckpt-weather-stream

# Define Query
q_weather = (
    weather_stream                       
      .select("*")                       # Columns to display
      .writeStream                       # Start defining the streaming sink
      .queryName("weather_stream")       
      .outputMode("append")              # No Aggregation            
      .format("console")                 
      .option("truncate", "false")             
      .option("numRows", 30)            
      .option("checkpointLocation", "file:///tmp/ckpt-weather-stream")  
      .trigger(processingTime="5 seconds")
      .start()
)

# Stop after 15 seconds
time.sleep(15)
q_weather.stop()

This query processes raw records from the producer (`weather.csv`) into the `weather_stream`.
  
*  `console`: Specifies that the output should be printed to the Spark console (inside the Docker container).

*  `append`: Specifies the output mode. Only new rows added since the last trigger are shown.
  
Since this query is used for debugging or monitoring the processing results, it is important to stop any existing query with the same name and clear the previous checkpoint directory at the beginning of re-run. This ensures that the output shown in the console is fresh, up-to-date, and reflects the current logic and data, rather than reusing cached or outdated state from earlier runs.

***Preview from Console***  
  
```
-------------------------------------------

Batch: 1

-------------------------------------------

+-------+-------------------+---------------+--------------+---------------+------------------+--------------+----------+----------+----------+-------------------+----+

|site_id|timestamp          |air_temperature|cloud_coverage|dew_temperature|sea_level_pressure|wind_direction|wind_speed|ts_send   |weather_ts|weather_ts_iso     |hour|

+-------+-------------------+---------------+--------------+---------------+------------------+--------------+----------+----------+----------+-------------------+----+
|0      |2022-01-16 15:00:00|18.9           |2             |15.6           |1013.1            |190           |4.1       |1761628447|1761628447|2025-10-28 05:14:07|15  |  
  
|0      |2022-01-16 16:00:00|21.1           |2             |15.6           |1013.2            |220           |5.1       |1761628447|1761628447|2025-10-28 05:14:07|16  |  
  
...

  
|0      |2022-01-17 05:00:00|17.8           |NULL          |13.3           |1009.6            |180           |4.1       |1761628448|1761628448|2025-10-28 05:14:07|5   |  
  
+-------+-------------------+---------------+--------------+---------------+------------------+--------------+----------+----------+----------+-------------------+----+  
  
only showing top 30 rows
```

This results shows that the query and code process successfully and accurate as the requirement.

### Load the new building information CSV file into a dataframe. Then, the data frames should be transformed into the proper formats following the metadata file schema, similar to assignment 2A.

In [10]:
# Load Static Building Information (predefined schema) 
building_df = (
    spark.read
         .option("header", "true")
         .option("mode", "FAILFAST")
         .schema(building_information_schema)
         .csv("./new_building_information.csv")
)

# Preview Schema
building_df.printSchema()

root
 |-- site_id: integer (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- primary_use: string (nullable = true)
 |-- square_feet: integer (nullable = true)
 |-- floor_count: integer (nullable = true)
 |-- row_id: integer (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- latent_y: float (nullable = true)
 |-- latent_s: float (nullable = true)
 |-- latent_r: float (nullable = true)



## 4.	Use a watermark on weather_ts, if data points are received 5 seconds late, discard the data.

In [17]:
# Event-time from epoch
weather_stream_ev = weather_stream \
    .withColumn("event_time", to_timestamp(from_unixtime(F.col("weather_ts")))) \
    .withWatermark("event_time", "5 seconds")

The code converts `weather_ts` ***from Unix format into a proper timestamp as `event_time`***, then applies a 5-second watermark to define the maximum allowed lateness for incoming records. This ensures that any ***data arriving more than 5 seconds late (based on `event_time`)*** will be ***excluded*** from downstream stateful operations.

***Sanity Check***

In [269]:
# Data Schema
weather_stream_ev.printSchema()

root
 |-- site_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- air_temperature: float (nullable = true)
 |-- cloud_coverage: integer (nullable = true)
 |-- dew_temperature: float (nullable = true)
 |-- sea_level_pressure: float (nullable = true)
 |-- wind_direction: integer (nullable = true)
 |-- wind_speed: float (nullable = true)
 |-- ts_send: integer (nullable = true)
 |-- weather_ts: integer (nullable = true)
 |-- weather_ts_iso: timestamp (nullable = true)
 |-- hour: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [270]:
# Streaming Connectivity
print("weather_stream_ev.isStreaming =", weather_stream_ev.isStreaming)

weather_stream_ev.isStreaming = True


The dataframe is streaming-enable which can receive data in real-time as long as the Producer send their records.

***Pull Query***

In [271]:
# Stop Similar Query for Re-Running
for q in spark.streams.active:
    if q.name == "weather_5s":
        q.stop()
        
# Clear the old checkpoint folder
!rm -rf /tmp/ckpt-weather-5s

# Define Query
q_weather_5s = (
    weather_stream_ev                       
      .select("*")                       # Columns to display
      .writeStream                       # Start defining the streaming sink
      .queryName("weather_5s")       
      .outputMode("append")              # No Aggregation           
      .format("console")                 
      .option("truncate", "false")             
      .option("numRows", 30)            
      .option("checkpointLocation", "file:///tmp/ckpt-weather-5s")  
      .trigger(processingTime="5 seconds")
      .start()
)

# Stop after 15 seconds
time.sleep(15)
q_weather_5s.stop()

Similar steps are implemented in this query. This code defines a streaming query (`weather_5s`) that outputs cleaned records to the console every 5 seconds.  
  
It applies a 5-second watermark based on event_time to drop late data and uses `.outputMode("append")` since ***no aggregation is involved***.  
  
Before restarting the query, it safely stops any active instance and clears the previous checkpoint to avoid conflicts.

***Preview from Console***  

```
-------------------------------------------

Batch: 1

-------------------------------------------

+-------+-------------------+---------------+--------------+---------------+------------------+--------------+----------+----------+----------+-------------------+----+-------------------+

|site_id|timestamp          |air_temperature|cloud_coverage|dew_temperature|sea_level_pressure|wind_direction|wind_speed|ts_send   |weather_ts|weather_ts_iso     |hour|event_time         |

+-------+-------------------+---------------+--------------+---------------+------------------+--------------+----------+----------+----------+-------------------+----+-------------------+

|0      |2022-01-08 00:00:00|19.4           |8             |15.0           |1014.7            |80            |2.6       |1761635718|1761635718|2025-10-28 07:15:18|0   |2025-10-28 18:15:18|

|0      |2022-01-08 01:00:00|17.8           |NULL          |15.6           |1015.1            |80            |3.1       |1761635718|1761635718|2025-10-28 07:15:18|1   |2025-10-28 18:15:18|

...

|0      |2022-01-09 04:00:00|16.1           |NULL          |15.0           |1013.1            |10            |2.1       |1761635719|1761635719|2025-10-28 07:15:18|4   |2025-10-28 18:15:19|

|0      |2022-01-09 05:00:00|15.0           |NULL          |14.4           |1012.9            |350           |2.1       |1761635719|1761635719|2025-10-28 07:15:18|5   |2025-10-28 18:15:19|

+-------+-------------------+---------------+--------------+---------------+------------------+--------------+----------+----------+----------+-------------------+----+-------------------+

only showing top 30 rows
```

The query successfully ingests streaming records from `weather_stream`, converts the epoch timestamp into `event_time`, and applies a 5-second watermark to filter out late data.  
  
As confirmed by the streaming console output, each batch correctly registers a watermark, ensuring that only timely records are processed in downstream tasks.

## 5.	Perform the necessary transformation you used in A2A. (note: every student may have used different features, feel free to reuse the code you have written in A2A. If you built an end-to-end pipeline, you can ignore this task.) 

### ***Pipeline Function (Transformation & Training)***

In [18]:
# Pipeline Function
def build_pipeline(train_df, label_col="energy_6h"):
    # Feature Groups
    categorical_SI = ["primary_use", "interval", "site_id", "building_id"]
    numeric_cols = [
        "square_feet", "floor_count", "year_built",           
        "latent_y", "latent_s", "latent_r",                   
        "avg_air_temperature_6h", "avg_dew_temperature_6h",   
        "avg_sea_level_pressure_6h", "avg_wind_speed_6h"
    ]
    
    # Available Columns
    available_cols = train_df.columns
    si_cols = [col for col in categorical_SI if col in available_cols]
    num_cols = [col for col in numeric_cols if col in available_cols]

    # String Indexers
    indexers = [
        StringIndexer(inputCol=col, outputCol=f"{col}_SI").setHandleInvalid("keep")
        for col in si_cols
    ]
    
    # Assembler
    inputCols = num_cols + [f"{col}_SI" for col in si_cols]
    assembler = VectorAssembler(inputCols=inputCols, outputCol="features")
    
    # Model
    gbt = GBTRegressor(
        featuresCol="features",
        labelCol=label_col,
        predictionCol="gbt_prediction",
        maxBins=2048
    ).setSeed(2025)    
    
    # Pipeline
    stages = indexers + [assembler, gbt]
    pipeline = Pipeline(stages=stages)

    return pipeline   

This end-to-end pipeline builds on the data transformation and model training framework developed in Assignment 2A.  

*  ***Data Transformation***: `StringIndexer` to encode categorical identifiers (e.g. `site_id`, `building_id`, etc) into numerical form suitable for model input.

*  ***Assembler***: Combined all transformed and numerical features into a single feature vector for model training.

*  ***Model Training***: Employed a Gradient-Boosted Tree (GBT) model, which previously outperformed the Random Forest (RF) model for this dataset in both accuracy and generalization.

### ***Data Preparation***

#### ***Training - Batch Transformation***

In [24]:
meters_df.printSchema()

root
 |-- building_id: integer (nullable = true)
 |-- meter_type: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- value: float (nullable = true)
 |-- row_id: integer (nullable = true)



In [19]:
# 1. Meters
meters_df = (
    meters_df
        .withColumn("hour", F.hour("ts"))
        .withColumn("date", F.to_date("ts"))
        .withColumn(
            "interval",
            F.when((F.col("hour") < 6), "0:00-5:59")
             .when((F.col("hour") < 12), "6:00-11:59")
             .when((F.col("hour") < 18), "12:00-17:59")
             .otherwise("18:00-23:59")
        )
)

# 6-hour Total Energy
building_energy = (
    meters_df.groupBy("building_id", "date", "interval")
             .agg(F.round(F.sum("value"), 2).alias("energy_6h"))
)

building_energy.show(5)

+-----------+----------+-----------+---------+
|building_id|      date|   interval|energy_6h|
+-----------+----------+-----------+---------+
|        233|2022-01-01|  0:00-5:59|    51.35|
|        889|2022-01-01|  0:00-5:59|  1960.27|
|       1104|2022-01-01| 6:00-11:59|111217.69|
|       1162|2022-01-01|12:00-17:59|  2822.64|
|       1218|2022-01-02| 6:00-11:59| 22076.42|
+-----------+----------+-----------+---------+
only showing top 5 rows



In [20]:
# 2. Weather Static
weather_static_df = (
    weather_static_df
        .withColumn("hour", F.hour("timestamp"))
        .withColumn("date", F.to_date("timestamp"))
        .withColumn(
            "interval",
            F.when((F.col("hour") < 6), "0:00-5:59")
             .when((F.col("hour") < 12), "6:00-11:59")
             .when((F.col("hour") < 18), "12:00-17:59")
             .otherwise("18:00-23:59")
        )
)

# 6-hour Weather Statistics
weather_static_agg = (
    weather_static_df.groupBy("site_id", "date", "interval")
        .agg(
            
            F.round(F.avg("air_temperature"), 2).alias("avg_air_temperature_6h"),
            F.round(F.avg("dew_temperature"), 2).alias("avg_dew_temperature_6h"),
            F.round(F.avg("sea_level_pressure"), 2).alias("avg_sea_level_pressure_6h"),
            F.round(F.avg("wind_speed"), 2).alias("avg_wind_speed_6h"),
            F.round(F.first("cloud_coverage"), 2).alias("mode_cloud_coverage_6h"),
            F.round(F.first("wind_direction"), 2).alias("mode_wind_direction_6h")
        )
        # Renamed Columns to Avoid Ambiguity
        .withColumnRenamed("site_id", "weather_site_id")
        .withColumnRenamed("date", "weather_date")
        .withColumnRenamed("interval", "weather_interval")
)

# Display Preview
weather_static_agg.show(2, truncate=False,vertical=True)

-RECORD 0--------------------------------
 weather_site_id           | 0           
 weather_date              | 2022-03-16  
 weather_interval          | 18:00-23:59 
 avg_air_temperature_6h    | 29.63       
 avg_dew_temperature_6h    | 16.03       
 avg_sea_level_pressure_6h | 1015.12     
 avg_wind_speed_6h         | 4.02        
 mode_cloud_coverage_6h    | NULL        
 mode_wind_direction_6h    | NULL        
-RECORD 1--------------------------------
 weather_site_id           | 0           
 weather_date              | 2022-05-20  
 weather_interval          | 18:00-23:59 
 avg_air_temperature_6h    | 27.32       
 avg_dew_temperature_6h    | 19.15       
 avg_sea_level_pressure_6h | 1016.88     
 avg_wind_speed_6h         | 5.07        
 mode_cloud_coverage_6h    | NULL        
 mode_wind_direction_6h    | NULL        
only showing top 2 rows



In [21]:
# Combine All Datasets
# Building Energy with Building Info
building_energy_info = building_energy.join(building_df, ["building_id"], "left")

building_energy_info.show(2, truncate=False,vertical=True)

-RECORD 0-----------------
 building_id | 233        
 date        | 2022-01-01 
 interval    | 0:00-5:59  
 energy_6h   | 51.35      
 site_id     | 2          
 primary_use | Education  
 square_feet | 25139      
 floor_count | 1          
 row_id      | 234        
 year_built  | 1977       
 latent_y    | 23.0       
 latent_s    | 4.400348   
 latent_r    | 3.0        
-RECORD 1-----------------
 building_id | 889        
 date        | 2022-01-01 
 interval    | 0:00-5:59  
 energy_6h   | 1960.27    
 site_id     | 9          
 primary_use | Education  
 square_feet | 126077     
 floor_count | 1          
 row_id      | 890        
 year_built  | 1991       
 latent_y    | 9.0        
 latent_s    | 5.100636   
 latent_r    | 4.0        
only showing top 2 rows



In [22]:
# Weather with Building Energy Info
feature_df = (
    weather_static_agg.join(
        building_energy_info,
        (weather_static_agg.weather_site_id == building_energy_info.site_id) &
        (weather_static_agg.weather_date == building_energy_info.date) &
        (weather_static_agg.weather_interval == building_energy_info.interval),
        "left"
    )
    .drop("weather_site_id", "weather_date", "weather_interval")
)

feature_df.show(2, truncate=False,vertical=True)

-RECORD 0--------------------------------
 avg_air_temperature_6h    | 29.63       
 avg_dew_temperature_6h    | 16.03       
 avg_sea_level_pressure_6h | 1015.12     
 avg_wind_speed_6h         | 4.02        
 mode_cloud_coverage_6h    | NULL        
 mode_wind_direction_6h    | NULL        
 building_id               | 14          
 date                      | 2022-03-16  
 interval                  | 18:00-23:59 
 energy_6h                 | 15364.57    
 site_id                   | 0           
 primary_use               | Education   
 square_feet               | 86250       
 floor_count               | 1           
 row_id                    | 15          
 year_built                | 2020        
 latent_y                  | 20.0        
 latent_s                  | 4.935759    
 latent_r                  | 1.0         
-RECORD 1--------------------------------
 avg_air_temperature_6h    | 29.63       
 avg_dew_temperature_6h    | 16.03       
 avg_sea_level_pressure_6h | 1015.

In [61]:
feature_df.toPandas()[['mode_cloud_coverage_6h','mode_wind_direction_6h']].value_counts()

Series([], Name: count, dtype: int64)

The columns `mode_cloud_coverage_6h` and `mode_wind_direction_6h` were originally computed using `F.first(...)`, which does not return the statistical mode but simply the first record per group. Due to missing values and unsorted input data, the result was that these columns were entirely null.  
  
To simplify the pipeline, these columns were dropped, as they contributed no meaningful information to the feature set.

In [23]:
# Drop null columns
feature_df = feature_df.drop("mode_cloud_coverage_6h", "mode_wind_direction_6h")

# Clean remaining nulls
feature_df = feature_df.dropna()

# Display Review
feature_df.show(2, truncate=False, vertical=True)

-RECORD 0--------------------------------
 avg_air_temperature_6h    | 20.93       
 avg_dew_temperature_6h    | 13.35       
 avg_sea_level_pressure_6h | 1017.78     
 avg_wind_speed_6h         | 3.85        
 building_id               | 42          
 date                      | 2022-01-02  
 interval                  | 18:00-23:59 
 energy_6h                 | 0.0         
 site_id                   | 0           
 primary_use               | Other       
 square_feet               | 226506      
 floor_count               | 1           
 row_id                    | 43          
 year_built                | 1975        
 latent_y                  | 25.0        
 latent_s                  | 5.3550797   
 latent_r                  | 2.0         
-RECORD 1--------------------------------
 avg_air_temperature_6h    | 20.93       
 avg_dew_temperature_6h    | 13.35       
 avg_sea_level_pressure_6h | 1017.78     
 avg_wind_speed_6h         | 3.85        
 building_id               | 75   

In [63]:
feature_df.printSchema()

root
 |-- avg_air_temperature_6h: double (nullable = true)
 |-- avg_dew_temperature_6h: double (nullable = true)
 |-- avg_sea_level_pressure_6h: double (nullable = true)
 |-- avg_wind_speed_6h: double (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- interval: string (nullable = true)
 |-- energy_6h: double (nullable = true)
 |-- site_id: integer (nullable = true)
 |-- primary_use: string (nullable = true)
 |-- square_feet: integer (nullable = true)
 |-- floor_count: integer (nullable = true)
 |-- row_id: integer (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- latent_y: float (nullable = true)
 |-- latent_s: float (nullable = true)
 |-- latent_r: float (nullable = true)



This section focuses on reproducing the data preparation (feature engineering) process from Assignment 2A, with a streamlined approach that only includes data aggregated at 6-hour intervals. Since the current task does not require extensive feature engineering, only initial transformations relevant to the 6-hour aggregation are retained.

##### ***Train Model***

In [64]:
# Extract a small static sample from the streaming source for training
gbt_pipeline = build_pipeline(feature_df)
gbt_model = gbt_pipeline.fit(feature_df)
gbt_model.write().overwrite().save("models/gbt_pipeline_model")

The model is trained using the static dataset (`feature_df`), which is then saved and later used to generate predictions on the incoming streaming data.

#### ***Streaming Path (Live Prediction)***

##### ***Weather 6h Aggregation***

In [24]:
# 1) Prep (keep one watermark only)
weather_stream_agg_prep = (
    weather_stream_ev
        .withColumn("date", F.to_date("timestamp"))
        .withColumn(
            "interval",
            F.when(F.col("hour") < 6,  "0:00-5:59")
             .when(F.col("hour") < 12, "6:00-11:59")
             .when(F.col("hour") < 18, "12:00-17:59")
             .otherwise("18:00-23:59")
        )
        .withWatermark("event_time", "5 seconds")   
)

# 2) 6-hour aggregation
weather_stream_agg = (
    weather_stream_agg_prep
        .groupBy(
            F.col("site_id"),
            F.col("date").alias("weather_date"),      
            F.col("interval"),
            F.window(F.col("event_time"), "6 hours").alias("w")
        )
        .agg(
            F.round(F.avg("air_temperature"), 2).alias("avg_air_temperature_6h"),
            F.round(F.avg("dew_temperature"), 2).alias("avg_dew_temperature_6h"),
            F.round(F.avg("sea_level_pressure"), 2).alias("avg_sea_level_pressure_6h"),
            F.round(F.avg("wind_speed"), 2).alias("avg_wind_speed_6h"),
            F.first("cloud_coverage", ignorenulls=True).alias("mode_cloud_coverage_6h"),
            F.first("wind_direction", ignorenulls=True).alias("mode_wind_direction_6h")
        )
        .withColumn("event_time", F.col("w").getField("end"))
        .drop("w")
        .withColumnRenamed("site_id", "weather_site_id")
        .withColumnRenamed("interval", "weather_interval")
)

Watermarking is reapplied because ***Spark does not retain watermark metadata across transformations*** such as `.select()`, `.withColumn()`, or `.groupBy().agg().select()`, which are applied during the 5-second late-data enforcement step (tumbling window).  
  
Even though the column `event_time` is still present, Spark ***no longer recognizes*** it as a watermarked event-time column, so to apply late-data filtering in subsequent operations (e.g., 6-hour window aggregation), the ***watermark must be explicitly redeclared***.  

***Sanity Check***

In [273]:
# Data Schema
weather_stream_agg.printSchema()

root
 |-- weather_site_id: integer (nullable = true)
 |-- weather_date: date (nullable = true)
 |-- weather_interval: string (nullable = false)
 |-- avg_air_temperature_6h: double (nullable = true)
 |-- avg_dew_temperature_6h: double (nullable = true)
 |-- avg_sea_level_pressure_6h: double (nullable = true)
 |-- avg_wind_speed_6h: double (nullable = true)
 |-- mode_cloud_coverage_6h: integer (nullable = true)
 |-- mode_wind_direction_6h: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [274]:
# Streaming Connectivity
print("weather_stream_agg.isStreaming =", weather_stream_agg.isStreaming)

weather_stream_agg.isStreaming = True


***Pull Query***

In [282]:
# Stop Similar Query for Re-Running
for q in spark.streams.active:
    if q.name == "weather_6h":
        q.stop()
        
# Clear the old checkpoint folder
!rm -rf /tmp/ckpt-weather-6h

# Define Query
q_weather_6h = (
    weather_stream_agg
      .writeStream
      .queryName("weather_6h")
      .outputMode("update")                 
      .format("console")
      .option("truncate", "false")
      .option("numRows", 30)
      .option("checkpointLocation", "file:///tmp/ckpt-weather-6h")
      .trigger(processingTime="5 seconds")
      .start()
)


# Stop after 15 seconds
time.sleep(15)
q_weather_6h.stop()

Similar steps are implemented in this query. This code defines a streaming query (`weather_6h`) that outputs 6-hour aggregated weather data to the console every 5 seconds.
  
It uses `.outputMode("complete")` since ***aggregation is involved***, allowing Spark to **reprint the full updated result set*** at each trigger.
  
Before restarting the query, it safely stops any active instance and clears the previous checkpoint to ensure the output reflects the most recent computation.

***Preview from Console***
   
```
-------------------------------------------

Batch: 1

-------------------------------------------

+---------------+------------+----------------+------------------------------------------+----------------------+----------------------+-------------------------+-----------------+----------------------+----------------------+

|weather_site_id|weather_date|weather_interval|w                                         |avg_air_temperature_6h|avg_dew_temperature_6h|avg_sea_level_pressure_6h|avg_wind_speed_6h|mode_cloud_coverage_6h|mode_wind_direction_6h|

+---------------+------------+----------------+------------------------------------------+----------------------+----------------------+-------------------------+-----------------+----------------------+----------------------+

|0              |2025-10-28  |12:00-17:59     |{2025-10-28 17:00:00, 2025-10-28 23:00:00}|26.56                 |23.01                 |1008.35                  |6.22             |6                     |10                    |

|0              |2025-10-28  |6:00-11:59      |{2025-10-28 17:00:00, 2025-10-28 23:00:00}|24.63                 |22.91                 |1008.3                   |5.07             |8                     |70                    |

|0              |2025-10-28  |18:00-23:59     |{2025-10-28 17:00:00, 2025-10-28 23:00:00}|27.14                 |22.59                 |1007.92                  |6.67             |6                     |360                   |

|0              |2025-10-28  |0:00-5:59       |{2025-10-28 17:00:00, 2025-10-28 23:00:00}|25.41                 |22.72                 |1011.26                  |4.52             |6                     |70                    |

+---------------+------------+----------------+------------------------------------------+----------------------+----------------------+-------------------------+-----------------+----------------------+----------------------+
```

The query successfully ingests streaming records from `weather_stream_ev`, add time grouping into  `weather_stream_agg_prep`, and finally done 6h-aggregation to generate `weather_stream_agg`.  
   
As confirmed by the streaming console output, each batch correctly shown the 6h aggregation and its `interval`.

##### ***Prediction Datasets***

In [25]:
# Join with Static Datasets
w = weather_stream_agg.alias("w")
b = broadcast(building_energy_info).alias("b")

feature_stream = (
    w.join(
        b,
        (F.col("w.weather_site_id")  == F.col("b.site_id"))   &
        (F.col("w.weather_date")     == F.col("b.date"))      &
        (F.col("w.weather_interval") == F.col("b.interval")),
        "left"
    )
    .drop("weather_site_id", "weather_date", "weather_interval")
)

# Drop null columns
feature_stream = feature_stream.drop("mode_cloud_coverage_6h", "mode_wind_direction_6h")

# Clean remaining nulls
feature_stream = feature_stream.dropna()

***Sanity Check***

In [288]:
# Data Schema
feature_stream.printSchema()

root
 |-- avg_air_temperature_6h: double (nullable = true)
 |-- avg_dew_temperature_6h: double (nullable = true)
 |-- avg_sea_level_pressure_6h: double (nullable = true)
 |-- avg_wind_speed_6h: double (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- interval: string (nullable = true)
 |-- energy_6h: double (nullable = true)
 |-- site_id: integer (nullable = true)
 |-- primary_use: string (nullable = true)
 |-- square_feet: integer (nullable = true)
 |-- floor_count: integer (nullable = true)
 |-- row_id: integer (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- latent_y: float (nullable = true)
 |-- latent_s: float (nullable = true)
 |-- latent_r: float (nullable = true)



In [289]:
# Streaming Connectivity
print("feature_stream.isStreaming =", feature_stream.isStreaming)

feature_stream.isStreaming = True


***Pull Query***

In [39]:
# Stop Similar Query for Re-Running
for q in spark.streams.active:
    if q.name == "feature_stream":
        q.stop()
        
# Clear the old checkpoint folder
!rm -rf /tmp/ckpt-feature-stream

# Define Query
q_feature_stream = (
    feature_stream                       
      .select("*")                       # Columns to display
      .writeStream                       # Start defining the streaming sink
      .queryName("feature_stream")       
      .outputMode("update")              # Aggregation          
      .format("console")                 
      .option("truncate", "false")             
      .option("numRows", 30)            
      .option("checkpointLocation", "file:///tmp/ckpt-feature-stream")  
      .trigger(processingTime="5 seconds") 
      .start()
)

# Stop after 60 seconds
time.sleep(60)
q_feature_stream.stop()

***Preview from Console***  

```
-------------------------------------------

Batch: 1

-------------------------------------------

+----------------------+----------------------+-------------------------+-----------------+-------------------+-----------+----------+-----------+---------+-------+-----------+-----------+-----------+------+----------+--------+---------+--------+

|avg_air_temperature_6h|avg_dew_temperature_6h|avg_sea_level_pressure_6h|avg_wind_speed_6h|event_time         |building_id|date      |interval   |energy_6h|site_id|primary_use|square_feet|floor_count|row_id|year_built|latent_y|latent_s |latent_r|

+----------------------+----------------------+-------------------------+-----------------+-------------------+-----------+----------+-----------+---------+-------+-----------+-----------+-----------+------+----------+--------+---------+--------+

|11.03                 |4.15                  |1022.6                   |3.0              |2025-10-29 17:00:00|83         |2022-01-14|0:00-5:59  |0.0      |0      |Other      |2070       |1          |84    |2003      |3.0     |3.3159704|2.0     |



...



|20.0                  |10.0                  |1018.77                  |3.77             |2025-10-29 17:00:00|19         |2022-01-14|18:00-23:59|0.0      |0      |Office     |18717      |1          |20    |2011      |11.0    |4.2722363|1.0     |

+----------------------+----------------------+-------------------------+-----------------+-------------------+-----------+----------+-----------+---------+-------+-----------+-----------+-----------+------+----------+--------+---------+--------+

only showing top 30 rows
```

The `feature_stream` is successfully stream to the kafka.

## 6.	Load your pipeline model and perform the following aggregations:    

### a)	Print the prediction from your model as a stream comes in.  

***Prediction***

In [26]:
# 6a
# Load trained pipeline model
gbt_model = PipelineModel.load("models/gbt_pipeline_model")

# Apply model on incoming stream
predictions_df = (
    gbt_model
        .transform(feature_stream)
        .select("site_id", "building_id", "date", "interval", "energy_6h",
                "gbt_prediction", "event_time")
)

***Sanity Check***

In [153]:
# Data Schema
predictions_df.printSchema()

root
 |-- site_id: integer (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- interval: string (nullable = true)
 |-- energy_6h: double (nullable = true)
 |-- gbt_prediction: double (nullable = false)
 |-- event_time: timestamp (nullable = true)



In [154]:
# Sanity Check
print("predictions_df.isStreaming =", predictions_df.isStreaming)

predictions_df.isStreaming = True


***Pull Query***

In [155]:
# Stop Similar Query for Re-Running
for q in spark.streams.active:
    if q.name == "prediction":
        q.stop()
        
# Clear the old checkpoint folder
!rm -rf /tmp/ckpt-prediction

# Define Query
q_prediction = (
    predictions_df                       
      .select("*")                       # Columns to display
      .writeStream                       # Start defining the streaming sink
      .queryName("prediction")       
      .outputMode("update")              # Transformation          
      .format("console")                 
      .option("truncate", "false")             
      .option("numRows", 30)            
      .option("checkpointLocation", "file:///tmp/ckpt-prediction")  
      .trigger(processingTime="5 seconds")
      .start()
)

# Stop after 60 seconds
time.sleep(60)
q_prediction.stop()

***Preview from Console***

```
-------------------------------------------

Batch: 1

-------------------------------------------

+-------+-----------+----------+-----------+---------+------------------+-------------------+

|site_id|building_id|date      |interval   |energy_6h|gbt_prediction    |event_time         |

+-------+-----------+----------+-----------+---------+------------------+-------------------+

|0      |96         |2022-02-16|12:00-17:59|48.12    |12251.389267191786|2025-10-29 23:00:00|



...


|0      |14         |2022-02-15|18:00-23:59|0.0      |7128.512323168765 |2025-10-29 23:00:00|

+-------+-----------+----------+-----------+---------+------------------+-------------------+

only showing top 30 rows
```

The `predictions_df` is successfully stream to Kafka with the expected results.

### b)	Every 7 seconds, print the total energy consumption for each 6-hour interval, aggregated by building, and print 20 records. (Note: This is simulating energy data each day in a week)  

In [27]:
# 6h-Energy
prediction_agg = (
    predictions_df
      .groupBy("site_id", "building_id", "date", "interval")
      .agg(
          F.sum("gbt_prediction").alias("total_predicted_energy"),
          F.sum("energy_6h").alias("total_actual_energy")
      )
)

***Sanity Check***

In [165]:
# Data Schema
prediction_agg.printSchema()

root
 |-- site_id: integer (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- interval: string (nullable = true)
 |-- total_predicted_energy: double (nullable = true)
 |-- total_actual_energy: double (nullable = true)



In [166]:
# Streaming Connectivity
print("prediction_agg.isStreaming =", prediction_agg.isStreaming)

prediction_agg.isStreaming = True


***Pull Query***

In [33]:
spark.conf.set("spark.sql.streaming.statefulOperator.checkCorrectness.enabled", "false")

In [168]:
# Stop Similar Query for Re-Running
for q in spark.streams.active:
    if q.name == "agg_pred_energy":
        q.stop()
        
# Clear the old checkpoint folder
!rm -rf /tmp/ckpt-energy-agg

# Define Query
q_pred_energy_agg = (
    prediction_agg.writeStream
        .queryName("agg_pred_energy")
        .outputMode("update")          # Aggregation
        .format("console")
        .option("truncate", "false")
        .option("numRows", 20)
        .option("checkpointLocation", "file:///tmp/ckpt-energy-agg")
        .trigger(processingTime="7 seconds")
        .start()
)

# Stop after 60 seconds
time.sleep(60)
q_pred_energy_agg.stop()

***Preview from Console***

```
-------------------------------------------

Batch: 1

-------------------------------------------

+-------+-----------+----------+-----------+----------------------+-------------------+

|site_id|building_id|date      |interval   |total_predicted_energy|total_actual_energy|

+-------+-----------+----------+-----------+----------------------+-------------------+

|0      |14         |2022-04-01|0:00-5:59  |7826.773218395835     |4811.98            |

|0      |4          |2022-03-29|12:00-17:59|6206.6363983930605    |0.0                |

...


|0      |25         |2022-03-29|6:00-11:59 |1658.1047339820698    |0.0                |

|0      |97         |2022-03-29|12:00-17:59|16253.553730405032    |12156.59           |

+-------+-----------+----------+-----------+----------------------+-------------------+

only showing top 20 rows
```

The `prediction_agg` that aggregated 6h-energy based on `building_id` and `interval` successfully streamed.

### c)	Every 14 seconds, for each site, print the daily total energy consumption.

In [28]:
# Daily energy consumption per site
daily_energy_agg = (
    predictions_df 
        .groupBy("site_id", "date")
        .agg(
            F.sum("gbt_prediction").alias("predicted_daily_energy"),
            F.sum("energy_6h").alias("actual_daily_energy")
        )
)

***Sanity Check***

In [171]:
# Data Schema
daily_energy_agg.printSchema()

root
 |-- site_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- predicted_daily_energy: double (nullable = true)
 |-- actual_daily_energy: double (nullable = true)



In [29]:
# Stream Connectivity
print("daily_energy_agg.isStreaming =", daily_energy_agg.isStreaming)

daily_energy_agg.isStreaming = True


***Pull Query***

In [217]:
# Stop Similar Query for Re-Running
for q in spark.streams.active:
    if q.name == "agg_daily_energy":
        q.stop()
        
# Clear the old checkpoint folder
!rm -rf /tmp/ckpt-energy-daily

# Define Query
q_pred_energy_daily = (
    daily_energy_agg.writeStream
        .queryName("agg_daily_energy")
        .outputMode("update")          # Aggregation
        .format("console")
        .option("truncate", "false")
        .option("numRows", 20)
        .option("checkpointLocation", "file:///tmp/ckpt-energy-daily")
        .trigger(processingTime="14 seconds")
        .start()
)

# Stop after 60 seconds
time.sleep(60)
q_pred_energy_daily.stop()

***Preview from Console***
```
-------------------------------------------

Batch: 1

-------------------------------------------

+-------+----------+----------------------+-------------------+

|site_id|date      |predicted_daily_energy|actual_daily_energy|

+-------+----------+----------------------+-------------------+

|0      |2022-04-26|494440.3804175244     |538005.81          |

|0      |2022-05-01|692467.6146897128     |945316.28          |

|0      |2022-04-28|639370.5065432675     |654017.3           |

|0      |2022-04-29|759138.1595306799     |745894.41          |

|0      |2022-05-02|777211.6180820164     |801354.4000000001  |

|0      |2022-04-30|774131.1098966111     |727029.03          |

|0      |2022-04-27|605610.1310638326     |583861.42          |

+-------+----------+----------------------+-------------------+
```

The `daily_energy_agg` that aggregated 6h-energy based on `site_id` and `date` successfully streamed.

## 7.	Save the data from 6 to Parquet files as streams.
  
(Hint: Parquet files support streaming writing/reading. The file keeps updating while new batches arrive.)

In [89]:
# Enable dynamic overwrite for partitioned Parquet output
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

In [122]:
# Clean previous output and checkpoints
shutil.rmtree("/tmp/parquet/pred_parquet",         ignore_errors=True)
shutil.rmtree("/tmp/parquet/6h_energy_parquet",    ignore_errors=True)
shutil.rmtree("/tmp/parquet/daily_energy_parquet", ignore_errors=True)
shutil.rmtree("/tmp/ckpt-pred-parquet",            ignore_errors=True)
shutil.rmtree("/tmp/ckpt-6h-energy-parquet",       ignore_errors=True)
shutil.rmtree("/tmp/ckpt-daily-energy-parquety",   ignore_errors=True)

# Reusable writer for batched overwrite with partitioning
def make_parquet_upserter(path, part_cols):
    def _writer(batch_df, epoch_id):
        (
            batch_df
            .write
            .mode("overwrite")
            .format("parquet")
            .partitionBy(*part_cols)
            .save(path)
        )
    return _writer

# Stream 6a: Save raw predictions (stateless) to Parquet
q_prediction = (
    predictions_df.writeStream
      .queryName("prediction_to_parquet_upsert")
      .outputMode("update")  
      .foreachBatch(make_parquet_upserter(
          "file:///tmp/parquet/pred_parquet",
          ["site_id", "building_id", "date", "interval"]
      ))
      .option("checkpointLocation", "file:///tmp/ckpt-pred-parquet")
      .trigger(processingTime="5 seconds")
      .start()
)

# Stream 6b: Aggregated prediction per interval to Parquet (update mode)
q_pred_energy_agg = (
    prediction_agg.writeStream
      .queryName("agg_pred_energy_to_parquet")
      .outputMode("update")
      .foreachBatch(make_parquet_upserter(
          "file:///tmp/parquet/6h_energy_parquet",
          ["site_id","building_id","date","interval"]
      ))
      .option("checkpointLocation", "file:///tmp/ckpt-6h-energy-parquet")
      .trigger(processingTime="7 seconds")
      .start()
)

# Stream 6c: Daily energy totals to Parquet
q_pred_energy_daily = (
    daily_energy_agg.writeStream
      .queryName("agg_daily_energy_to_parquet")
      .outputMode("update")
      .foreachBatch(make_parquet_upserter(
          "file:///tmp/parquet/daily_energy_parquet",
          ["site_id","date"]
      ))
      .option("checkpointLocation", "file:///tmp/ckpt-daily-energy-parquety")
      .trigger(processingTime="14 seconds")
      .start()
)

# Run 5 minutes for see the data Demo 
time.sleep(300)
for q in spark.streams.active:
       q.stop()

This script configures Spark for dynamic partition overwrite and clears previous output and checkpoint directories to ensure a clean start. It then defines three streaming write pipelines:  
   
(1) `q_prediction` writes raw prediction results (stateless) to Parquet in append mode every 5 seconds;  
  
(2) `q_pred_energy_agg` aggregates predictions by site, building, date, and interval, then writes updates to partitioned Parquet files using foreachBatch every 7 seconds; and  
  
(3) `q_pred_energy_daily` computes daily energy totals and writes them similarly.  
  
A reusable make_parquet_upserter function ensures each batch overwrites only the relevant partitions.  
  
The script runs all streams for 5 minutes before stopping them gracefully for the checking of the data saved.

***Preview Results***

In [38]:
# Define schemas
predictions_schema = StructType([
    StructField("site_id", IntegerType()),
    StructField("building_id", IntegerType()),
    StructField("date", DateType()),
    StructField("interval", StringType()),
    StructField("gbt_prediction", DoubleType()),
    StructField("event_time", TimestampType())
])

energy_agg_schema = StructType([
    StructField("site_id", IntegerType()),
    StructField("building_id", IntegerType()),
    StructField("date", DateType()),
    StructField("interval", StringType()),
    StructField("total_predicted_energy", DoubleType()),
    StructField("total_actual_energy", DoubleType())
])

daily_energy_schema = StructType([
    StructField("site_id", IntegerType()),
    StructField("date", DateType()),
    StructField("predicted_daily_energy", DoubleType()),
    StructField("actual_daily_energy", DoubleType())
])

***6a. Prediction***

In [123]:
# Read Parquet Files
predictions_check = spark.read.schema(predictions_schema).parquet("/tmp/parquet/pred_parquet")

# Display Records
predictions_check.show(5, truncate=False)

# Data Schema
predictions_check.printSchema()

+------------------+-------------------+-------+-----------+----------+----------+
|gbt_prediction    |event_time         |site_id|building_id|date      |interval  |
+------------------+-------------------+-------+-----------+----------+----------+
|2161.0318108739098|2025-10-30 23:00:00|0      |42         |2022-03-02|6:00-11:59|
|128.15161758897256|2025-10-30 23:00:00|0      |94         |2022-03-10|0:00-5:59 |
|2449.5553186631128|2025-10-30 23:00:00|0      |42         |2022-03-05|0:00-5:59 |
|3503.1481218823787|2025-10-30 23:00:00|0      |4          |2022-03-02|6:00-11:59|
|381.67883646402595|2025-10-30 23:00:00|0      |64         |2022-03-04|6:00-11:59|
+------------------+-------------------+-------+-----------+----------+----------+
only showing top 5 rows

root
 |-- gbt_prediction: double (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- site_id: integer (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- interval:

In [124]:
# Count total rows
row_count = predictions_check.count()
print(f"Total rows in predictions Parquet: {row_count}")

Total rows in predictions Parquet: 953


***6b. 6h Energy per Building***

In [125]:
# Read Parquet Files
energy_agg_check = spark.read.schema(energy_agg_schema).parquet("file:///tmp/parquet/6h_energy_parquet")

# Display Records
energy_agg_check.show(5, truncate=False)

# Data Schema
energy_agg_check.printSchema()

+----------------------+-------------------+-------+-----------+----------+-----------+
|total_predicted_energy|total_actual_energy|site_id|building_id|date      |interval   |
+----------------------+-------------------+-------+-----------+----------+-----------+
|1104.9881665454086    |0.0                |0      |57         |2022-03-09|12:00-17:59|
|6274.615694610532     |11433.17           |0      |96         |2022-03-04|0:00-5:59  |
|9476.493776957446     |8906.37            |0      |91         |2022-03-08|18:00-23:59|
|491.85943419477945    |0.0                |0      |19         |2022-03-10|0:00-5:59  |
|12006.641141780317    |13254.06           |0      |14         |2022-03-09|12:00-17:59|
+----------------------+-------------------+-------+-----------+----------+-----------+
only showing top 5 rows

root
 |-- total_predicted_energy: double (nullable = true)
 |-- total_actual_energy: double (nullable = true)
 |-- site_id: integer (nullable = true)
 |-- building_id: integer (nullab

In [247]:
# Count total rows
row_count = energy_agg_check.count()
print(f"Total rows in 6h_energy Parquet: {row_count}")

Total rows in 6h_energy Parquet: 1061


***6c. Daily Energy per Site***

In [127]:
# Read Parquet Files
daily_energy_check = spark.read.schema(daily_energy_schema).parquet("file:///tmp/parquet/daily_energy_parquet")

# Display Records
daily_energy_check.show(5, truncate=False)

# Data Schema
daily_energy_check.printSchema()

+----------------------+-------------------+-------+----------+
|predicted_daily_energy|actual_daily_energy|site_id|date      |
+----------------------+-------------------+-------+----------+
|647669.4890899977     |538396.03          |0      |2022-03-13|
|425438.9827481691     |430560.52          |0      |2022-03-02|
|347232.5758826666     |320012.77          |0      |2022-03-08|
|310491.6755882687     |352139.98          |0      |2022-03-03|
|549892.9119280739     |466948.75          |0      |2022-03-10|
+----------------------+-------------------+-------+----------+
only showing top 5 rows

root
 |-- predicted_daily_energy: double (nullable = true)
 |-- actual_daily_energy: double (nullable = true)
 |-- site_id: integer (nullable = true)
 |-- date: date (nullable = true)



In [128]:
# Count total rows
row_count = daily_energy_check.count()
print(f"Total rows in daily_energy Parquet: {row_count}")

Total rows in daily_energy Parquet: 12


All the parquet files are check and shown the expected records and schema.

## 8. Read the parquet files from task 7 as data streams and send them to Kafka topics with appropriate names.
  
(Note: You shall read the parquet files as a streaming data frame and send messages to the Kafka topic when new data appears in the parquet file.)

In [254]:
# Stream Parquet files written in Task 7
predictions_stream = spark.readStream.schema(predictions_schema).format("parquet").load("file:///tmp/parquet/pred_parquet")
energy_agg_stream   = spark.readStream.schema(energy_agg_schema).format("parquet").load("file:///tmp/parquet/6h_energy_parquet")
daily_energy_stream = spark.readStream.schema(daily_energy_schema).format("parquet").load("file:///tmp/parquet/daily_energy_parquet")

# Convert to Kafka-compatible format
predictions_kafka = predictions_stream.select(
    concat_ws("_", "site_id", "building_id").alias("key"),
    to_json(struct("*")).alias("value")
)

energy_agg_kafka = energy_agg_stream.select(
    concat_ws("_", "site_id", "building_id", "date", "interval").alias("key"),
    to_json(struct("*")).alias("value")
)

daily_energy_kafka = daily_energy_stream.select(
    concat_ws("_", "site_id", "date").alias("key"),
    to_json(struct("*")).alias("value")
)
       
# Define Kafka topics for each stream
predictions_topic = "predictions5s"
energy_agg_topic = "energyagg7s"
daily_energy_topic = "dailyenergy14s"

***Individual Run for Streaming***

In [ ]:
# 6a -> 7a
# Clear old output
shutil.rmtree("/tmp/parquet/pred_parquet", ignore_errors=True)
shutil.rmtree("/tmp/ckpt-pred-parquet", ignore_errors=True)
shutil.rmtree("/tmp/ckpt-kafka-predictions", ignore_errors=True)

# Stream 6a: Save raw predictions (stateless) to Parquet
q_prediction = (
    predictions_df.writeStream
      .queryName("prediction_to_parquet_upsert")
      .outputMode("update")  
      .foreachBatch(make_parquet_upserter(
          "file:///tmp/parquet/pred_parquet",
          ["site_id", "building_id", "date", "interval"]
      ))
      .option("checkpointLocation", "file:///tmp/ckpt-pred-parquet")
      .trigger(processingTime="5 seconds")
      .start()
)

q_kafka_predictions = (
    predictions_kafka.writeStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{hostip}:9092")
        .option("topic", predictions_topic)
        .option("checkpointLocation", "/tmp/ckpt-kafka-predictions")
        .outputMode("append")
        .trigger(processingTime="5 seconds")
        .queryName("to_kafka_predictions")
        .start()
)

In [273]:
# 6b -> 7b
# Clear old output
shutil.rmtree("/tmp/parquet/6h_energy_parquet", ignore_errors=True)
shutil.rmtree("/tmp/ckpt-6h-energy-parquet", ignore_errors=True)
shutil.rmtree("/tmp/ckpt-kafka-energy6h", ignore_errors=True)

# Stream 6b: Aggregated prediction per interval to Parquet (update mode)
q_pred_energy_agg = (
    prediction_agg.writeStream
      .queryName("agg_pred_energy_to_parquet")
      .outputMode("update")
      .foreachBatch(make_parquet_upserter(
          "file:///tmp/parquet/6h_energy_parquet",
          ["site_id","building_id","date","interval"]
      ))
      .option("checkpointLocation", "file:///tmp/ckpt-6h-energy-parquet")
      .trigger(processingTime="7 seconds")
      .start()
)

q_kafka_energy_agg = (
    energy_agg_kafka.writeStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{hostip}:9092")
        .option("topic", energy_agg_topic)
        .option("checkpointLocation", "/tmp/ckpt-kafka-energy6h")
        .outputMode("append")
        .trigger(processingTime="7 seconds")
        .queryName("to_kafka_energy6h")
        .start()
)

In [283]:
# 6c -> 7c
# Clear old output
shutil.rmtree("/tmp/parquet/daily_energy_parquet", ignore_errors=True)
shutil.rmtree("/tmp/ckpt-daily-energy-parquety", ignore_errors=True)
shutil.rmtree("/tmp/ckpt-kafka-dailyenergy", ignore_errors=True)

# Stream 6c: Daily energy totals to Parquet
q_pred_energy_daily = (
    daily_energy_agg.writeStream
      .queryName("agg_daily_energy_to_parquet")
      .outputMode("update")
      .foreachBatch(make_parquet_upserter(
          "file:///tmp/parquet/daily_energy_parquet",
          ["site_id","date"]
      ))
      .option("checkpointLocation", "file:///tmp/ckpt-daily-energy-parquety")
      .trigger(processingTime="14 seconds")
      .start()
)

q_kafka_daily_energy = (
    daily_energy_kafka.writeStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{hostip}:9092")
        .option("topic", daily_energy_topic)
        .option("checkpointLocation", "/tmp/ckpt-kafka-dailyenergy")
        .outputMode("append")
        .trigger(processingTime="14 seconds")
        .queryName("to_kafka_dailyenergy")
        .start()
)

***Sanity Check***

In [284]:
# Data Sending
for q in spark.streams.active:
    print(q.name, q.status["message"])
    print(q.lastProgress)

agg_daily_energy_to_parquet Processing new data
None
to_kafka_dailyenergy Waiting for next trigger
{'id': '18cac180-1f1a-411d-945c-a5b3ecc3bd1d', 'runId': '4fa79043-0aa1-4d18-b6f1-628dab3747f2', 'name': 'to_kafka_dailyenergy', 'timestamp': '2025-10-30T09:40:08.656Z', 'batchId': 0, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'latestOffset': 1, 'triggerExecution': 1}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[file:/tmp/parquet/daily_energy_parquet]', 'startOffset': None, 'endOffset': None, 'latestOffset': None, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0}], 'sink': {'description': 'org.apache.spark.sql.kafka010.KafkaSourceProvider$KafkaTable@30700a0f', 'numOutputRows': -1}}


In [285]:
# Connectivitiy
print("Is q_kafka_predictions active?", q_kafka_predictions.isActive)
print("Is q_kafka_energy_agg active?", q_kafka_energy_agg.isActive)
print("Is q_kafka_daily_energy active?", q_kafka_daily_energy.isActive)

Is q_kafka_predictions active? False
Is q_kafka_energy_agg active? False
Is q_kafka_daily_energy active? True


***Manual Control***

In [286]:
spark.streams.active

In [287]:
# Stop All Queries
for q in spark.streams.active:
        q.stop()

## AI Acknowledgement

I acknowledge the use of AI in the following ways:
 
(i) Assist in some parts of the codes in which they raised errors.   
I used ChatGPT (chatgpt.com) to resolve some parts of my codes that did not run successfully.

(ii) Generate ideas on how to solve the problems  
I used ChatGPT (chatgpt.com) to ask about possible alternatives or ideas on how to answer some of the questions. For example, when answering questions, I asked ChatGPT to tell me how to approach the discussion from the result.  

(iii) Rewrite and/or rephrase a portion of the markdown text  
I used ChatGPT (chatgpt.com) to assist me rewriting and/or rephrasing the explanations or ideas behind my submitted codes in this assessment. First, I wrote a draft for specific parts, then I put them to ChatGPT to get the refined writings.